# 🛰️ Drone Fleet Tracker — Tactical Ops Dashboard

An advanced, real-time drone fleet monitoring dashboard built with **Dash** + **dash-leaflet**.

**Features**
- Live tactical map (dark theme) with animated drone markers rotated to true heading
- Per-drone flight path trails
- Color-coded status rings (battery / link health)
- Click any drone to open a live telemetry panel (altitude, battery, speed, heading compass)
- Simulated real-time telemetry stream (updates every 1.5s) — swap in a real WebSocket/MQTT feed later
- Runs entirely inside Colab, no external server needed

Run each cell top to bottom. The dashboard will render inline at the bottom.


## 1. Install dependencies

In [ ]:
!pip -q install dash==2.17.1 dash-leaflet==1.0.15 dash-extensions==1.0.18 numpy pandas


## 2. Imports & configuration

In [ ]:
import json
import math
import random
from collections import deque
from datetime import datetime, timezone

import numpy as np
import dash
from dash import dcc, html, Input, Output, State, callback_context, no_update
import dash_leaflet as dl
import plotly.graph_objects as go

random.seed(7)
np.random.seed(7)

# ---- Ops area configuration -------------------------------------------------
BASE_LAT, BASE_LON = 34.0522, -118.2437   # center of the AOI (swap for your own coords)
AOI_RADIUS_DEG = 0.12                      # rough patrol radius in degrees
TRAIL_LEN = 25                             # how many past points to keep per drone
TICK_MS = 1500                             # telemetry refresh interval

DRONE_NAMES = ["RAVEN-1", "RAVEN-2", "FALCON-3", "FALCON-4", "HAWK-5", "HAWK-6"]

STATUS_COLORS = {
    "nominal":  "#00ff9c",
    "caution":  "#ffb000",
    "critical": "#ff3b3b",
}


## 3. Drone fleet simulation\n\nEach drone follows a smooth pseudo-random patrol path around a waypoint, with altitude, speed, battery and heading evolving realistically over time. Replace `Fleet.tick()` with a real telemetry feed (WebSocket/MQTT/REST) to go live.

In [ ]:
class Drone:
    def __init__(self, name, idx):
        self.name = name
        angle = (idx / len(DRONE_NAMES)) * 2 * math.pi
        self.lat = BASE_LAT + AOI_RADIUS_DEG * 0.5 * math.sin(angle)
        self.lon = BASE_LON + AOI_RADIUS_DEG * 0.5 * math.cos(angle)
        self.heading = random.uniform(0, 360)
        self.altitude = random.uniform(300, 800)      # ft AGL
        self.speed = random.uniform(18, 35)            # m/s
        self.battery = random.uniform(70, 100)          # %
        self.link_quality = random.uniform(80, 100)      # %
        self.wp_angle = angle
        self.wp_radius = AOI_RADIUS_DEG * random.uniform(0.6, 1.0)
        self.wp_speed = random.uniform(0.008, 0.02) * random.choice([-1, 1])
        self.trail = deque(maxlen=TRAIL_LEN)
        self.trail.append((self.lat, self.lon))
        self.status = "nominal"

    def tick(self):
        # advance along a slowly rotating circular patrol path, with a little jitter
        self.wp_angle += self.wp_speed
        target_lat = BASE_LAT + self.wp_radius * math.sin(self.wp_angle)
        target_lon = BASE_LON + self.wp_radius * math.cos(self.wp_angle)

        dlat = target_lat - self.lat
        dlon = target_lon - self.lon
        self.heading = (math.degrees(math.atan2(dlon, dlat)) + 360) % 360

        self.lat += dlat * 0.15 + np.random.normal(0, 0.0005)
        self.lon += dlon * 0.15 + np.random.normal(0, 0.0005)
        self.trail.append((self.lat, self.lon))

        self.altitude = float(np.clip(self.altitude + np.random.normal(0, 12), 150, 1200))
        self.speed = float(np.clip(self.speed + np.random.normal(0, 1.2), 5, 45))
        self.battery = float(np.clip(self.battery - random.uniform(0.02, 0.08), 0, 100))
        self.link_quality = float(np.clip(self.link_quality + np.random.normal(0, 2), 40, 100))

        if self.battery < 20 or self.link_quality < 55:
            self.status = "critical"
        elif self.battery < 40 or self.link_quality < 75:
            self.status = "caution"
        else:
            self.status = "nominal"

    def to_dict(self):
        return {
            "name": self.name,
            "lat": self.lat, "lon": self.lon,
            "heading": self.heading,
            "altitude": self.altitude,
            "speed": self.speed,
            "battery": self.battery,
            "link_quality": self.link_quality,
            "status": self.status,
            "trail": list(self.trail),
        }


class Fleet:
    def __init__(self):
        self.drones = {name: Drone(name, i) for i, name in enumerate(DRONE_NAMES)}

    def tick(self):
        for d in self.drones.values():
            d.tick()
        return {name: d.to_dict() for name, d in self.drones.items()}


fleet = Fleet()
print(f"Fleet online: {len(fleet.drones)} UAS assets")


## 4. Drone map icon\n\nA rotated SVG divIcon so each marker visually points along its true heading, with a status-colored ring.

In [ ]:
def drone_icon_html(heading, color):
    return f"""
    <div style="transform: rotate({heading}deg); width:34px; height:34px;">
      <svg viewBox="0 0 24 24" width="34" height="34">
        <circle cx="12" cy="12" r="11" fill="rgba(10,14,20,0.55)"
                stroke="{color}" stroke-width="1.6"/>
        <path d="M12 3 L16 15 L12 12.5 L8 15 Z" fill="{color}"/>
      </svg>
    </div>
    """

def make_marker(d):
    return dl.Marker(
        position=[d["lat"], d["lon"]],
        icon={
            "html": drone_icon_html(d["heading"], STATUS_COLORS[d["status"]]),
            "className": "drone-divicon",
            "iconSize": [34, 34],
            "iconAnchor": [17, 17],
        },
        id={"type": "drone-marker", "name": d["name"]},
        children=[dl.Tooltip(d["name"])],
    )

def make_trail(d):
    return dl.Polyline(
        positions=d["trail"],
        color=STATUS_COLORS[d["status"]],
        weight=2, opacity=0.55, dashArray="4,5",
    )


## 5. App layout — dark tactical theme

In [ ]:
app = dash.Dash(__name__)
app.title = "Drone Fleet Tracker"

DARK = "#0a0e14"
PANEL = "#0f1620"
BORDER = "#1c2733"
ACCENT = "#00fff2"
TEXT_DIM = "#7d8ea3"

app.index_string = '''
<!DOCTYPE html>
<html>
<head>
{%metas%}
<title>{%title%}</title>
{%favicon%}
{%css%}
<link href="https://fonts.googleapis.com/css2?family=Share+Tech+Mono&family=Rajdhani:wght@500;700&display=swap" rel="stylesheet">
<style>
  body { background:#0a0e14; margin:0; font-family:'Rajdhani',sans-serif; }
  .drone-divicon { filter: drop-shadow(0 0 4px rgba(0,255,242,0.5)); }
  .leaflet-container { background:#0a0e14 !important; }
  ::-webkit-scrollbar { width:6px; }
  ::-webkit-scrollbar-thumb { background:#1c2733; }
</style>
</head>
<body>
  {%app_entry%}
  <footer>{%config%}{%scripts%}{%renderer%}</footer>
</body>
</html>
'''

def stat_bar(label, value, unit, color, max_val):
    pct = max(0, min(100, (value / max_val) * 100))
    return html.Div([
        html.Div([
            html.Span(label, style={"color": TEXT_DIM, "fontSize": "11px", "letterSpacing": "1px"}),
            html.Span(f"{value:.1f}{unit}", style={"color": color, "fontSize": "12px", "float": "right", "fontFamily": "'Share Tech Mono', monospace"}),
        ]),
        html.Div(style={
            "height": "6px", "background": "#141c26", "borderRadius": "3px",
            "marginTop": "4px", "overflow": "hidden",
        }, children=html.Div(style={
            "width": f"{pct}%", "height": "100%", "background": color,
            "boxShadow": f"0 0 8px {color}", "transition": "width 0.6s ease",
        }))
    ], style={"marginBottom": "14px"})


app.layout = html.Div(style={"background": DARK, "minHeight": "100vh", "color": "#d8e3ee", "padding": "16px"}, children=[

    dcc.Interval(id="tick", interval=TICK_MS, n_intervals=0),
    dcc.Store(id="fleet-store"),
    dcc.Store(id="selected-drone", data=DRONE_NAMES[0]),

    # header
    html.Div(style={
        "display": "flex", "justifyContent": "space-between", "alignItems": "center",
        "borderBottom": f"1px solid {BORDER}", "paddingBottom": "10px", "marginBottom": "14px",
    }, children=[
        html.Div([
            html.Span("◆ ", style={"color": ACCENT}),
            html.Span("UAS FLEET TRACKER", style={"fontWeight": "700", "fontSize": "22px", "letterSpacing": "3px"}),
            html.Span("  //  SECTOR AOI-7", style={"color": TEXT_DIM, "fontSize": "13px", "marginLeft": "10px"}),
        ]),
        html.Div(id="clock", style={"fontFamily": "'Share Tech Mono', monospace", "color": ACCENT, "fontSize": "14px"}),
    ]),

    html.Div(style={"display": "flex", "gap": "14px"}, children=[

        # map panel
        html.Div(style={
            "flex": "3", "background": PANEL, "border": f"1px solid {BORDER}",
            "borderRadius": "8px", "overflow": "hidden", "height": "620px", "position": "relative",
        }, children=[
            dl.Map(
                id="map",
                center=[BASE_LAT, BASE_LON], zoom=12,
                style={"width": "100%", "height": "100%"},
                children=[
                    dl.TileLayer(url="https://{s}.basemaps.cartocdn.com/dark_all/{z}/{x}/{y}{r}.png",
                                 attribution="&copy; CartoDB"),
                    dl.LayerGroup(id="trail-layer"),
                    dl.LayerGroup(id="marker-layer"),
                ]
            ),
            html.Div("LIVE FEED", style={
                "position": "absolute", "top": "10px", "right": "14px",
                "background": "rgba(10,14,20,0.75)", "border": f"1px solid {BORDER}",
                "padding": "4px 10px", "borderRadius": "4px", "fontSize": "11px",
                "color": STATUS_COLORS["nominal"], "letterSpacing": "2px",
            }),
        ]),

        # fleet list + telemetry panel
        html.Div(style={"flex": "1.3", "display": "flex", "flexDirection": "column", "gap": "14px"}, children=[

            html.Div(style={
                "background": PANEL, "border": f"1px solid {BORDER}", "borderRadius": "8px", "padding": "12px",
            }, children=[
                html.Div("FLEET ROSTER", style={"color": TEXT_DIM, "fontSize": "11px", "letterSpacing": "2px", "marginBottom": "8px"}),
                html.Div(id="roster-list"),
            ]),

            html.Div(id="telemetry-panel", style={
                "background": PANEL, "border": f"1px solid {BORDER}", "borderRadius": "8px",
                "padding": "14px", "flex": "1",
            }),
        ]),
    ]),
])


## 6. Callbacks — simulation tick, map render, click-to-inspect

In [ ]:
@app.callback(Output("fleet-store", "data"), Input("tick", "n_intervals"))
def update_fleet(n):
    return fleet.tick()


@app.callback(Output("clock", "children"), Input("tick", "n_intervals"))
def update_clock(n):
    return datetime.now(timezone.utc).strftime("%H:%M:%S UTC")


@app.callback(
    Output("marker-layer", "children"),
    Output("trail-layer", "children"),
    Input("fleet-store", "data"),
)
def render_map(data):
    if not data:
        return no_update, no_update
    markers = [make_marker(d) for d in data.values()]
    trails = [make_trail(d) for d in data.values()]
    return markers, trails


@app.callback(
    Output("selected-drone", "data"),
    Input({"type": "drone-marker", "name": dash.ALL}, "n_clicks"),
    State({"type": "drone-marker", "name": dash.ALL}, "id"),
    prevent_initial_call=True,
)
def select_drone(n_clicks, ids):
    ctx = callback_context
    if not ctx.triggered or not any(n_clicks):
        return no_update
    triggered_id = json.loads(ctx.triggered[0]["prop_id"].split(".")[0])
    return triggered_id["name"]


@app.callback(Output("roster-list", "children"), Input("fleet-store", "data"), Input("selected-drone", "data"))
def render_roster(data, selected):
    if not data:
        return no_update
    rows = []
    for name, d in data.items():
        active = name == selected
        color = STATUS_COLORS[d["status"]]
        rows.append(html.Div(
            [
                html.Span("● ", style={"color": color}),
                html.Span(name, style={"fontFamily": "'Share Tech Mono', monospace", "fontSize": "13px"}),
                html.Span(d["status"].upper(), style={
                    "float": "right", "fontSize": "10px", "color": color, "letterSpacing": "1px",
                }),
            ],
            id={"type": "roster-row", "name": name},
            n_clicks=0,
            style={
                "padding": "7px 8px", "borderRadius": "4px", "cursor": "pointer",
                "background": "rgba(0,255,242,0.08)" if active else "transparent",
                "border": f"1px solid {'#00fff2' if active else 'transparent'}",
                "marginBottom": "4px",
            },
        ))
    return rows


@app.callback(
    Output("selected-drone", "data", allow_duplicate=True),
    Input({"type": "roster-row", "name": dash.ALL}, "n_clicks"),
    State({"type": "roster-row", "name": dash.ALL}, "id"),
    prevent_initial_call=True,
)
def select_from_roster(n_clicks, ids):
    ctx = callback_context
    if not ctx.triggered or not any(n_clicks):
        return no_update
    triggered_id = json.loads(ctx.triggered[0]["prop_id"].split(".")[0])
    return triggered_id["name"]


def compass(heading, color):
    fig = go.Figure()
    fig.add_trace(go.Scatterpolar(
        r=[0, 1], theta=[heading, heading], mode="lines",
        line=dict(color=color, width=3), hoverinfo="skip",
    ))
    fig.update_layout(
        polar=dict(
            bgcolor="rgba(0,0,0,0)",
            radialaxis=dict(visible=False, range=[0, 1]),
            angularaxis=dict(direction="clockwise", rotation=90, gridcolor="#1c2733",
                              tickfont=dict(color="#7d8ea3", size=9)),
        ),
        showlegend=False, height=170, margin=dict(l=10, r=10, t=10, b=10),
        paper_bgcolor="rgba(0,0,0,0)",
    )
    return fig


@app.callback(
    Output("telemetry-panel", "children"),
    Input("fleet-store", "data"),
    Input("selected-drone", "data"),
)
def render_telemetry(data, selected):
    if not data or selected not in data:
        return "No asset selected."
    d = data[selected]
    color = STATUS_COLORS[d["status"]]
    return [
        html.Div([
            html.Span(d["name"], style={"fontWeight": "700", "fontSize": "17px", "letterSpacing": "1px"}),
            html.Span(f'  {d["status"].upper()}', style={"color": color, "fontSize": "12px", "marginLeft": "8px"}),
        ], style={"marginBottom": "12px", "borderBottom": f"1px solid {BORDER}", "paddingBottom": "8px"}),

        stat_bar("BATTERY", d["battery"], "%", color, 100),
        stat_bar("LINK QUALITY", d["link_quality"], "%", "#00fff2", 100),
        stat_bar("ALTITUDE", d["altitude"], " ft", "#00fff2", 1200),
        stat_bar("SPEED", d["speed"], " m/s", "#00fff2", 45),

        html.Div("HEADING", style={"color": TEXT_DIM, "fontSize": "11px", "letterSpacing": "1px", "marginTop": "6px"}),
        dcc.Graph(figure=compass(d["heading"], color), config={"displayModeBar": False}),

        html.Div([
            html.Span("POS  ", style={"color": TEXT_DIM, "fontSize": "11px"}),
            html.Span(f'{d["lat"]:.4f}, {d["lon"]:.4f}', style={"fontFamily": "'Share Tech Mono', monospace", "fontSize": "12px"}),
        ]),
    ]


## 7. Launch\n\nRenders inline in Colab. If the inline iframe doesn't appear, change `jupyter_mode` to `"external"` and open the printed URL.

In [ ]:
app.run(jupyter_mode="inline", jupyter_height=760, debug=False)


---
### Going further (ideas to extend this)
- **Real telemetry**: replace `Fleet.tick()` with a WebSocket/MQTT client pulling live position reports (e.g. MAVLink, ADS-B, or a custom REST feed) — the rest of the dashboard needs no changes.
- **Geofencing**: draw `dl.Polygon` no-fly zones and flag a drone red when its trail crosses the boundary.
- **Multi-sector view**: add a `dcc.Dropdown` to switch between several `Fleet` instances / AOIs.
- **Alerts feed**: log status transitions (nominal → caution → critical) into a scrolling `html.Div` ticker.
- **Persistence**: log each tick to a pandas DataFrame / SQLite for after-action review and flight-path playback.
